# Домашнее задание 1. Банковские счета

Дисциплина: язык Python для разработчиков.

Решение состоит из модуля `bank_account.py` и этого ноутбука.
Ноутбук показывает все три части задания:

1. Базовый класс `Account`: операции, инкапсуляция, история с датой и статусом.
2. Наследники `CheckingAccount` и `SavingsAccount`, проверка имени и крупные операции.
3. Загрузка общей истории из CSV и JSON и очистка ошибок методом `clean_history`.

Файлы `transactions_dirty.csv` и `transactions_dirty.json` общие для нескольких счетов.
В них есть опечатки (`diposit`, `wdraw`, `succes`), пустые и отрицательные суммы,
несуществующие даты и повторные строки. Такие записи в историю не попадают.


## Задание 1. Базовый счёт

Номер счёта создаётся сам в формате `ACC-XXXX`, отсчёт начинается с 1000.
Баланс хранится в приватном `_balance`, наружу его отдаёт `get_balance`.
Каждая операция — словарь с типом, суммой, датой, балансом после операции и статусом.
Нехватка денег не меняет баланс, но попытка записывается со статусом `fail`.
Отрицательная сумма вызывает исключение.


In [3]:
import importlib

import bank_account
from datetime import datetime

importlib.reload(bank_account)
from bank_account import Account, CheckingAccount, SavingsAccount

Account.reset_counter()

account = Account("Иван Иванов", 1000)
print(account)
print("Баланс:", account.get_balance())

account.deposit(250)
print("Снятие 50:", account.withdraw(50))
print("Снятие больше остатка:", account.withdraw(1000))
print("Баланс после неудачного снятия:", account.get_balance())
print("История:")
for line in account.get_history():
    print(line)

operation = account.operations_history[0]
print("Поля операции:", sorted(operation))
print("Дата — это datetime:", isinstance(operation["datetime"], datetime))


Account(holder='Иван Иванов', account_number='ACC-1000', balance=1000.00)
Баланс: 1000.0
Снятие 50: True
Снятие больше остатка: True
Баланс после неудачного снятия: 200.0
История:
2026-10-04 14:12:39 | deposit | сумма: 250.00 | баланс: 1250.00 | success
2026-10-04 14:12:39 | withdraw | сумма: 50.00 | баланс: 1200.00 | success
2026-10-04 14:12:39 | withdraw | сумма: 1000.00 | баланс: 200.00 | success
Поля операции: ['amount', 'balance', 'datetime', 'status', 'type']
Дата — это datetime: True


## Задание 2. Наследование, проверка данных и крупные операции

`CheckingAccount` — расчётный счёт, `SavingsAccount` — сберегательный.
Оба наследуют `Account` и хранят классовый атрибут `account_type`.

У сберегательного счёта есть `apply_interest(rate)`: `0.07` означает 7% на остаток.
Снять можно не больше 50% текущего баланса. Более крупная попытка остаётся в истории
со статусом `fail`.

Имя владельца принимается только в виде «Имя Фамилия» с заглавной буквы,
целиком на кириллице или целиком на латинице.
`get_large_operations` показывает последние n успешных операций не меньше заданного порога
и умеет ограничивать их периодом.


In [4]:
Account.reset_counter()
checking = CheckingAccount("Anna Smirnova", 200)
savings = SavingsAccount("Мария Петрова", 1000)

print(checking.account_number, checking.account_type, checking.get_balance())
print(savings.account_number, savings.account_type, savings.get_balance())
print("Наследование:", isinstance(checking, Account), isinstance(savings, Account))

print("Снятие ровно 50%:", savings.withdraw(500), "баланс", savings.get_balance())
print("Снятие больше 50%:", savings.withdraw(300), "баланс", savings.get_balance())
print("Начислено процентов:", savings.apply_interest(0.07))
print("Баланс после процентов:", savings.get_balance())

for bad_name in ("иван иванов", "Иван", "Иван Ivanov", "IVAN IVANOV"):
    try:
        Account(bad_name)
    except ValueError as error:
        print(f"{bad_name!r}: {error}")

try:
    checking.deposit(-20)
except ValueError as error:
    print("Отрицательное пополнение:", error)

print("Крупные операции сберегательного счёта:")
savings.deposit(2500)
savings.get_large_operations(2, min_amount=500)


ACC-1000 checking 200.0
ACC-1001 savings 1000.0
Наследование: True True
Снятие ровно 50%: True баланс 500.0
Снятие больше 50%: False баланс 500.0
Начислено процентов: 35.0
Баланс после процентов: 535.0
'иван иванов': Имя владельца должно быть в формате «Имя Фамилия» с заглавной буквы, кириллицей или латиницей.
'Иван': Имя владельца должно быть в формате «Имя Фамилия» с заглавной буквы, кириллицей или латиницей.
'Иван Ivanov': Имя и фамилия должны быть в одной письменности: либо кириллицей, либо латиницей.
'IVAN IVANOV': Имя владельца должно быть в формате «Имя Фамилия» с заглавной буквы, кириллицей или латиницей.
Отрицательное пополнение: Сумма должна быть положительной.
Крупные операции сберегательного счёта:
Последние 2 крупных операций (порог 500.00):
2026-10-04 14:12:54 | deposit | сумма: 2500.00 | баланс: 3035.00
2026-10-04 14:12:54 | withdraw | сумма: 500.00 | баланс: 500.00


['2026-10-04 14:12:54 | deposit | сумма: 2500.00 | баланс: 3035.00',
 '2026-10-04 14:12:54 | withdraw | сумма: 500.00 | баланс: 500.00']

## Задание 3. Грязные данные

Общий файл содержит операции нескольких счетов: `ACC-100001`, `ACC-100002`,
`ACC-100003` и `ACC-100004`. `load_history` оставляет строки одного номера,
затем `clean_history` отбрасывает ошибки и проводит оставшиеся операции.
Баланс после загрузки считается по принятым операциям.

Для расчётного счёта допустимы только `deposit` и `withdraw`.
Для сберегательного дополнительно допустим `interest`.
CSV и JSON содержат одни и те же строки.


In [5]:
import importlib

import bank_account
from datetime import datetime

importlib.reload(bank_account)
from bank_account import Account, CheckingAccount, SavingsAccount

accounts = (
    ("ACC-100001", CheckingAccount, "Иван Иванов"),
    ("ACC-100002", SavingsAccount, "Мария Петрова"),
    ("ACC-100003", CheckingAccount, "Пётр Сидоров"),
    ("ACC-100004", SavingsAccount, "Анна Смирнова"),
)

for path in ("transactions_dirty.csv", "transactions_dirty.json"):
    print("\nФайл:", path)
    loaded = []
    for number, cls, holder in accounts:
        account = cls(holder)
        stats = account.load_history(path, account_number=number)
        loaded.append(account)
        print(
            f"{holder} ({number}, {account.account_type}): "
            f"{stats}, баланс {account.get_balance()}"
        )

ivan = loaded[0]
print("\nИстория ACC-100001, Иван Иванов:")
for line in ivan.get_history():
    print(line)
print("Отклонено:")
for row in ivan.rejected_records:
    print(
        f"  {row.get('type')!r}, сумма {row.get('amount')!r}, "
        f"дата {row.get('datetime')!r}: {row['reason']}"
    )

maria = loaded[1]
print("\nКрупные операции ACC-100002:")
maria.get_large_operations(
    3,
    min_amount=400,
    date_from=datetime(2025, 9, 1),
    date_to=datetime(2025, 10, 31),
)



Файл: transactions_dirty.csv
Иван Иванов (ACC-100001, checking): {'loaded': 17, 'rejected': 9}, баланс 3664.0
Мария Петрова (ACC-100002, savings): {'loaded': 18, 'rejected': 9}, баланс 2883.24
Пётр Сидоров (ACC-100003, checking): {'loaded': 13, 'rejected': 8}, баланс 4646.0
Анна Смирнова (ACC-100004, savings): {'loaded': 19, 'rejected': 12}, баланс 8057.25

Файл: transactions_dirty.json
Иван Иванов (ACC-100001, checking): {'loaded': 17, 'rejected': 9}, баланс 3664.0
Мария Петрова (ACC-100002, savings): {'loaded': 18, 'rejected': 9}, баланс 2883.24
Пётр Сидоров (ACC-100003, checking): {'loaded': 13, 'rejected': 8}, баланс 4646.0
Анна Смирнова (ACC-100004, savings): {'loaded': 19, 'rejected': 12}, баланс 8057.25

История ACC-100001, Иван Иванов:
2025-09-27 22:17:26 | deposit | сумма: 921.00 | баланс: 921.00 | success
2025-09-27 22:17:26 | deposit | сумма: 607.00 | баланс: 1528.00 | success
2025-09-28 22:17:00 | deposit | сумма: 129.00 | баланс: 1657.00 | success
2025-09-28 22:17:26 | de

['2025-10-13 22:17:26 | withdraw | сумма: 440.00 | баланс: 3105.98',
 '2025-10-09 22:17:26 | deposit | сумма: 622.00 | баланс: 3517.72',
 '2025-10-09 22:17:00 | deposit | сумма: 638.00 | баланс: 3065.72']